# Dividend Adjustment Effects on Black–Scholes–Merton Option Pricing Accuracy
## Complete revised analysis — Google Colab / GitHub notebook

This notebook reproduces the **revised empirical analysis** developed in response to peer review for the paper on short-maturity U.S. equity call options.

### What this notebook does

- loads the 13 option-chain snapshots from **August 29–September 18, 2025**;
- audits stale quotes and the NVDA anomalies identified by the reviewers;
- defines the observed option premium using the **positive two-sided bid–ask midpoint**;
- estimates volatility **independently** from pre-sample underlying returns (HV10, HV15, HV20);
- constructs a cleaned primary sample using a uniform activity/quote screen;
- prices calls under BSM with `q=0` and BSM with `q>0`;
- evaluates the first-order dividend-sensitivity approximation;
- reports MAE, normalized MAE, signed error, paired daily tests, and Holm-adjusted p-values;
- runs robustness checks across volatility windows, quote-age thresholds, spread caps, and `lastPrice`;
- includes a Yahoo-implied-volatility diagnostic to demonstrate why IV-based fit is not treated as independent validation;
- includes a 100-step Cox–Ross–Rubinstein American-call diagnostic;
- generates the publication figures, including the **improved Figure 1** with clearly differentiated market points and model lines;
- exports all tables, figures, and the cleaned primary dataset.

### Data access

The notebook first looks for a `data/` folder in the current project. In Google Colab, if the data are not present, it tries to clone the public GitHub repository. If the required files are still unavailable, it prompts you to upload the companion `BSM_Colab_Data.zip` file.

**No option-chain or historical-price data are downloaded from Yahoo Finance at runtime.** The analysis uses the frozen data files included with the replication package/repository.


## 0. Environment and data setup

The setup cell is designed to work in three situations:

1. **Google Colab opened from GitHub** — the repository is cloned automatically if needed;
2. **Google Colab with the companion data ZIP** — upload `BSM_Colab_Data.zip` when prompted;
3. **Local Jupyter** — run the notebook from a repository root containing `data/`.


In [ ]:
from pathlib import Path
import os, re, sys, zipfile, subprocess, shutil, platform

IN_COLAB = 'google.colab' in sys.modules

# Install the one dependency that may not be present in a fresh Colab runtime.
if IN_COLAB:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'statsmodels>=0.14,<0.15'
    ])

GITHUB_REPO = 'https://github.com/abdallahmaiga10/BSM-option-pricing.git'
REPO_DIR = Path('/content/BSM-option-pricing') if IN_COLAB else Path.cwd()

REQUIRED_CALL_COUNT = 13
REQUIRED_HISTORY = 'historical_closes_presample.csv'


def is_valid_data_dir(data_dir: Path) -> bool:
    return (
        data_dir.exists()
        and len(list(data_dir.glob('calls-2025-*.csv'))) == REQUIRED_CALL_COUNT
        and (data_dir / REQUIRED_HISTORY).exists()
    )


def find_root_with_data(candidates):
    for root in candidates:
        root = Path(root)
        if is_valid_data_dir(root / 'data'):
            return root.resolve()
    return None

# First try common project locations.
candidates = [Path.cwd(), REPO_DIR, Path('/content')]
ROOT = find_root_with_data(candidates)

# In Colab, clone the GitHub repository if data are not already present.
if ROOT is None and IN_COLAB:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print('Required data not found locally. Trying to clone:')
    print(GITHUB_REPO)
    clone = subprocess.run(
        ['git', 'clone', '--depth', '1', GITHUB_REPO, str(REPO_DIR)],
        text=True, capture_output=True
    )
    if clone.returncode != 0:
        print('GitHub clone was not available; upload fallback will be used.')
    ROOT = find_root_with_data([REPO_DIR])

# Final Colab fallback: upload a ZIP or the 14 CSV files directly.
if ROOT is None and IN_COLAB:
    from google.colab import files
    BASE = Path('/content/bsm_revision_upload')
    BASE.mkdir(parents=True, exist_ok=True)
    print('\nUpload BSM_Colab_Data.zip (recommended), or upload all 13 calls CSV files')
    print('plus historical_closes_presample.csv.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No data were uploaded.')

    for name, blob in uploaded.items():
        dest = BASE / Path(name).name
        dest.write_bytes(blob)
        if zipfile.is_zipfile(dest):
            with zipfile.ZipFile(dest) as zf:
                zf.extractall(BASE)

    # Find a data/ directory inside extracted content.
    roots = [BASE]
    roots.extend([p.parent for p in BASE.rglob('data') if p.is_dir()])
    ROOT = find_root_with_data(roots)

    # Support direct CSV upload by collecting files into BASE/data/.
    if ROOT is None:
        direct_calls = list(BASE.glob('calls-2025-*.csv'))
        direct_hist = BASE / REQUIRED_HISTORY
        if len(direct_calls) == REQUIRED_CALL_COUNT and direct_hist.exists():
            data_dir = BASE / 'data'
            data_dir.mkdir(exist_ok=True)
            for p in direct_calls:
                shutil.copy2(p, data_dir / p.name)
            shutil.copy2(direct_hist, data_dir / direct_hist.name)
            ROOT = BASE

if ROOT is None:
    raise FileNotFoundError(
        'Could not find the required data. Run from a repository root containing data/, '
        'or use Google Colab and upload BSM_Colab_Data.zip.'
    )

DATA = ROOT / 'data'
print('Using replication root:', ROOT)
print('Option snapshot files:', len(list(DATA.glob('calls-2025-*.csv'))))
print('Historical-close file:', DATA / REQUIRED_HISTORY)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import statsmodels
from scipy.stats import norm, ttest_rel, wilcoxon
from statsmodels.stats.multitest import multipletests
from IPython.display import display

FIG = ROOT / 'revised_outputs' / 'figures'
TAB = ROOT / 'revised_outputs' / 'tables'
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

# Primary analysis constants
RISK_FREE = 0.0441
EXPIRY = pd.Timestamp('2025-09-19')
# Labor Day is the only exchange holiday relevant to the <=1-session screen in the sample window.
HOLIDAYS = np.array(['2025-09-01'], dtype='datetime64[D]')
TICKERS = ['AAPL', 'BAC', 'JPM', 'MRK', 'MSFT', 'NVDA', 'PFE', 'UNH', 'WFC']

print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('SciPy:', scipy.__version__)
print('statsmodels:', statsmodels.__version__)


## 1. Load the 13 option snapshots and construct quote/timing variables

For each contract-date observation we construct:

- the snapshot date from the filename;
- parsed `lastTradeDate` and calendar-day quote age;
- a business-session activity measure for the primary freshness screen;
- market midpoint `(bid + ask)/2`;
- relative bid–ask spread;
- stock price `S`, strike `K`, dividend yield `q`, Yahoo IV, ACT/ACT maturity, and normalized strike `K/S`.


In [ ]:
raws = []
for p in sorted(DATA.glob('calls-2025-*.csv')):
    d = pd.read_csv(p)
    date_match = re.search(r'(2025-\d{2}-\d{2})', p.name)
    if date_match is None:
        raise ValueError(f'Could not parse snapshot date from {p.name}')
    d['snapshot_date'] = pd.Timestamp(date_match.group(1))
    raws.append(d)

raw = pd.concat(raws, ignore_index=True)

raw['lastTradeDate_parsed'] = (
    pd.to_datetime(raw['lastTradeDate'], utc=True, errors='coerce')
      .dt.tz_convert(None)
)
raw['last_trade_date'] = raw['lastTradeDate_parsed'].dt.normalize()
raw['trade_age_calendar_days'] = (raw['snapshot_date'] - raw['last_trade_date']).dt.days

valid_last_trade = raw['last_trade_date'].notna()
trade_age_sessions = np.full(len(raw), np.nan)
trade_age_sessions[valid_last_trade.to_numpy()] = np.busday_count(
    raw.loc[valid_last_trade, 'last_trade_date'].values.astype('datetime64[D]'),
    raw.loc[valid_last_trade, 'snapshot_date'].values.astype('datetime64[D]'),
    holidays=HOLIDAYS
)
raw['trade_age_sessions'] = trade_age_sessions

raw['market_mid'] = (raw['bid'] + raw['ask']) / 2.0
raw['relative_spread'] = (raw['ask'] - raw['bid']) / raw['market_mid']
raw['S'] = pd.to_numeric(raw['stockPrice'], errors='coerce')
raw['K'] = pd.to_numeric(raw['strike'], errors='coerce')
raw['q'] = pd.to_numeric(raw['dividend_yield'], errors='coerce')
raw['yahoo_iv'] = pd.to_numeric(raw['impliedVolatility'], errors='coerce')
raw['T_actact'] = (EXPIRY - raw['snapshot_date']).dt.days / 365.0
raw['K_over_S'] = raw['K'] / raw['S']

print('Raw rows:', len(raw))
display(raw.groupby('symbol').size().rename('Raw N').to_frame())


## 2. Raw-data audit and NVDA diagnostic

This section documents the data-quality concern highlighted by both reviewers. The filtering rules used later are **ticker-neutral**; NVDA is not removed or treated separately.


In [ ]:
audit = raw.groupby('symbol').agg(
    raw_N=('symbol', 'size'),
    mean_age_days=('trade_age_calendar_days', 'mean'),
    median_age_days=('trade_age_calendar_days', 'median'),
    max_age_days=('trade_age_calendar_days', 'max'),
    older_than_7_days=('trade_age_calendar_days', lambda x: (x > 7).sum()),
    zero_or_negative_bid=('bid', lambda x: (x <= 0).sum()),
    zero_or_negative_ask=('ask', lambda x: (x <= 0).sum())
)
display(audit)

nv = raw[raw['symbol'] == 'NVDA']
nvda_audit = pd.Series({
    'NVDA raw rows': len(nv),
    'strike > 200': (nv['K'] > 200).sum(),
    'strike > 200 and last trade before 2024-06-10': (
        (nv['K'] > 200) & (nv['last_trade_date'] < pd.Timestamp('2024-06-10'))
    ).sum(),
    'max strike': nv['K'].max(),
    'mean Yahoo IV': nv['yahoo_iv'].mean(),
    'max Yahoo IV': nv['yahoo_iv'].max(),
    'zero Yahoo IV': (nv['yahoo_iv'] == 0).sum()
})
display(nvda_audit.to_frame('Value'))


## 3. Independent pre-sample historical volatility

The primary model does **not** use option-implied volatility as its volatility input. Instead, volatility is estimated from pre-sample daily underlying log returns. The manuscript uses **HV20** as the primary specification; HV10 and HV15 are retained as robustness checks.

For a window of `n` daily returns,

\[
\widehat{\sigma}=\sqrt{252}\;\mathrm{sd}(r_t),\qquad
r_t=\log(S_t/S_{t-1}).
\]


In [ ]:
hist = pd.read_csv(DATA / 'historical_closes_presample.csv', parse_dates=['date'])


def sigma_window(g, n):
    g = g.sort_values('date')
    log_returns = np.log(g['close'] / g['close'].shift(1)).dropna()
    if len(log_returns) < n:
        raise ValueError(f'Need at least {n} returns for {g.name if hasattr(g, "name") else "ticker"}.')
    return log_returns.iloc[-n:].std(ddof=1) * np.sqrt(252)


sigmas = {}
for n in [10, 15, 20]:
    sigmas[n] = {
        sym: sigma_window(g, n)
        for sym, g in hist.groupby('symbol')
    }

volatility_table = pd.DataFrame({
    'HV10': sigmas[10],
    'HV15': sigmas[15],
    'HV20': sigmas[20]
}).sort_index()

display(volatility_table)


## 4. Primary cleaned sample

The primary sample retains a contract-date observation only when:

1. bid and ask are both present and strictly positive;
2. `ask >= bid`;
3. `S > 0` and `K > 0`;
4. the positive midpoint does not exceed the stock price (`midpoint <= S`, the call upper bound);
5. `q >= 0` and maturity is positive;
6. `lastTradeDate` is available and is no more than **one business session** before the snapshot date.

The same screen is applied to every ticker.


In [ ]:
mask = (
    raw['bid'].notna()
    & raw['ask'].notna()
    & (raw['bid'] > 0)
    & (raw['ask'] > 0)
    & (raw['ask'] >= raw['bid'])
    & (raw['S'] > 0)
    & (raw['K'] > 0)
    & (raw['market_mid'] > 0)
    & (raw['market_mid'] <= raw['S'])
    & (raw['q'] >= 0)
    & (raw['T_actact'] > 0)
    & raw['trade_age_sessions'].notna()
    & (raw['trade_age_sessions'] >= 0)
    & (raw['trade_age_sessions'] <= 1)
)

prim = raw.loc[mask].copy()
prim['sigma_hv20'] = prim['symbol'].map(sigmas[20])

print('Primary rows:', len(prim))
display(prim.groupby('symbol').size().rename('Primary N').to_frame())


## 5. BSM prices and the analytical dividend-omission effect

For a European call with continuous dividend yield `q`,

\[
C(q)=Se^{-qT}N(d_1)-Ke^{-rT}N(d_2),
\]

where

\[
d_1=\frac{\ln(S/K)+(r-q+\tfrac12\sigma^2)T}{\sigma\sqrt{T}},
\qquad d_2=d_1-\sigma\sqrt{T}.
\]

The dividend sensitivity is

\[
\frac{\partial C}{\partial q}=-TSe^{-qT}N(d_1),
\]

so the first-order effect of incorrectly setting a positive dividend yield to zero is

\[
C(0)-C(q)\approx qTSe^{-qT}N(d_1).
\]


In [ ]:
def bsm_call(S, K, T, r, sigma, q=0.0):
    S = np.asarray(S, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    q = np.asarray(q, dtype=float)

    d1 = (
        np.log(S / K) + (r - q + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    return (
        S * np.exp(-q * T) * norm.cdf(d1)
        - K * np.exp(-r * T) * norm.cdf(d2)
    )


prim['BS_q0'] = bsm_call(
    prim['S'], prim['K'], prim['T_actact'],
    RISK_FREE, prim['sigma_hv20'], 0.0
)
prim['BSM_q'] = bsm_call(
    prim['S'], prim['K'], prim['T_actact'],
    RISK_FREE, prim['sigma_hv20'], prim['q']
)

# Identically defined errors for both specifications.
prim['signed_error_q0'] = prim['BS_q0'] - prim['market_mid']
prim['signed_error_q'] = prim['BSM_q'] - prim['market_mid']
prim['AE_q0'] = prim['signed_error_q0'].abs()
prim['AE_q'] = prim['signed_error_q'].abs()
prim['AE_improvement'] = prim['AE_q0'] - prim['AE_q']

# Exact and first-order dividend-omission effect.
prim['dividend_omission_effect'] = prim['BS_q0'] - prim['BSM_q']
d1q = (
    np.log(prim['S'] / prim['K'])
    + (RISK_FREE - prim['q'] + 0.5 * prim['sigma_hv20']**2) * prim['T_actact']
) / (prim['sigma_hv20'] * np.sqrt(prim['T_actact']))

prim['dividend_approx'] = (
    prim['q'] * prim['T_actact'] * prim['S']
    * np.exp(-prim['q'] * prim['T_actact'])
    * norm.cdf(d1q)
)
prim['approx_abs_error'] = (
    prim['dividend_omission_effect'] - prim['dividend_approx']
).abs()


## 6. Primary summary tables and paired daily tests

Dollar MAE is retained for within-ticker economic interpretation. For cross-ticker comparison the notebook also reports normalized MAE,

\[
\mathrm{NMAE}=rac{\sum_i |C_i^{model}-C_i^{market}|}{\sum_i C_i^{market}}.
\]

To avoid treating all strikes from an option chain as independent, inferential comparisons are performed on **13 paired ticker-day MAEs**. Both paired t-tests and Wilcoxon signed-rank tests are reported, together with Holm multiplicity adjustments across the nine tickers.


In [ ]:
sample_summary = []
pricing_summary = []
tests = []

for sym in TICKERS:
    g = prim[prim['symbol'] == sym]
    rg = raw[raw['symbol'] == sym]

    mae_q0 = g['AE_q0'].mean()
    mae_q = g['AE_q'].mean()

    sample_summary.append([
        sym,
        len(rg),
        len(g),
        g['trade_age_calendar_days'].mean(),
        100 * g['relative_spread'].mean(),
        g['market_mid'].mean(),
        100 * g['sigma_hv20'].iloc[0],
        100 * g['q'].mean()
    ])

    pricing_summary.append([
        sym,
        len(g),
        mae_q0,
        mae_q,
        g['AE_q0'].sum() / g['market_mid'].sum(),
        g['AE_q'].sum() / g['market_mid'].sum(),
        100 * (mae_q0 - mae_q) / mae_q0,
        g['signed_error_q0'].mean(),
        g['signed_error_q'].mean()
    ])

    daily = g.groupby('snapshot_date').agg(
        mae0=('AE_q0', 'mean'),
        maeq=('AE_q', 'mean')
    )
    diff = daily['mae0'] - daily['maeq']

    tests.append([
        sym,
        diff.mean(),
        ttest_rel(daily['mae0'], daily['maeq']).pvalue,
        wilcoxon(daily['mae0'], daily['maeq']).pvalue,
        (diff > 0).sum(),
        (diff < 0).sum(),
        len(diff)
    ])

sample_summary = pd.DataFrame(sample_summary, columns=[
    'Ticker', 'Raw N', 'Primary N', 'Mean age (calendar days)',
    'Mean spread %', 'Avg midpoint', 'HV20 %', 'Dividend yield %'
])

pricing_summary = pd.DataFrame(pricing_summary, columns=[
    'Ticker', 'N', 'MAE q=0', 'MAE q>0', 'NMAE q=0', 'NMAE q>0',
    'MAE improvement %', 'Signed error q=0', 'Signed error q>0'
])

tests = pd.DataFrame(tests, columns=[
    'Ticker', 'Mean daily MAE0-MAEq', 'Paired t p', 'Wilcoxon p',
    'Days q>0 better', 'Days q>0 worse', 'Days'
])

tests['Holm t p'] = multipletests(tests['Paired t p'], method='holm')[1]
tests['Holm Wilcoxon p'] = multipletests(tests['Wilcoxon p'], method='holm')[1]

display(sample_summary)
display(pricing_summary)
display(tests)

approx_corr = np.corrcoef(
    prim['dividend_omission_effect'], prim['dividend_approx']
)[0, 1]
mean_approx_abs_error = prim['approx_abs_error'].mean()

print(f'Approximation correlation: {approx_corr:.9f}')
print(f'Mean absolute approximation error: {mean_approx_abs_error:.9f}')


## 7. Robustness checks

The following robustness variants change one design choice at a time:

- historical-volatility window: HV10, HV15, HV20;
- activity screen: 0, 1, 2, 3, or 5 business sessions;
- relative bid–ask spread caps: 25%, 50%, 100%;
- recent `lastPrice` instead of the midpoint.


In [ ]:
def select_sample(max_sessions=1, spread_cap=None):
    m = (
        raw['bid'].notna()
        & raw['ask'].notna()
        & (raw['bid'] > 0)
        & (raw['ask'] > 0)
        & (raw['ask'] >= raw['bid'])
        & (raw['S'] > 0)
        & (raw['K'] > 0)
        & (raw['market_mid'] > 0)
        & (raw['market_mid'] <= raw['S'])
        & (raw['q'] >= 0)
        & (raw['T_actact'] > 0)
        & raw['trade_age_sessions'].notna()
        & (raw['trade_age_sessions'] >= 0)
        & (raw['trade_age_sessions'] <= max_sessions)
    )
    if spread_cap is not None:
        m &= raw['relative_spread'] <= spread_cap
    return raw.loc[m].copy()


def pricing_variant(d, hv_window=20, market='market_mid'):
    d = d.copy()
    d['sigma'] = d['symbol'].map(sigmas[hv_window])
    d = d[d[market].notna() & (d[market] > 0)]

    d['p0'] = bsm_call(d['S'], d['K'], d['T_actact'], RISK_FREE, d['sigma'], 0.0)
    d['pq'] = bsm_call(d['S'], d['K'], d['T_actact'], RISK_FREE, d['sigma'], d['q'])
    d['e0'] = (d['p0'] - d[market]).abs()
    d['eq'] = (d['pq'] - d[market]).abs()

    rows = []
    for sym in TICKERS:
        g = d[d['symbol'] == sym]
        mae0 = g['e0'].mean()
        maeq = g['eq'].mean()
        rows.append([
            sym, len(g),
            100 * (mae0 - maeq) / mae0
        ])
    return pd.DataFrame(rows, columns=['Ticker', 'N', 'MAE improvement %'])


robustness_parts = []

for n in [10, 15, 20]:
    z = pricing_variant(select_sample(1), hv_window=n)
    z['Specification'] = f'HV{n}, activity <=1 session'
    robustness_parts.append(z)

for sessions in [0, 1, 2, 3, 5]:
    z = pricing_variant(select_sample(sessions), hv_window=20)
    z['Specification'] = f'HV20, activity <={sessions} sessions'
    robustness_parts.append(z)

for cap in [0.25, 0.50, 1.00]:
    z = pricing_variant(select_sample(1, spread_cap=cap), hv_window=20)
    z['Specification'] = f'HV20, spread <={cap:.0%}'
    robustness_parts.append(z)

z = pricing_variant(select_sample(1), hv_window=20, market='lastPrice')
z['Specification'] = 'HV20, recent lastPrice'
robustness_parts.append(z)

robustness = pd.concat(robustness_parts, ignore_index=True)
robustness_pivot = robustness.pivot(
    index='Ticker', columns='Specification', values='MAE improvement %'
)
display(robustness_pivot.round(2))


## 8. Yahoo implied-volatility diagnostic — not primary validation

This diagnostic intentionally reproduces a specification using Yahoo's option-implied volatility. The resulting near-perfect market/model correlations are **not interpreted as independent evidence of model accuracy**, because implied volatility is itself recovered from option prices.


In [ ]:
ivd = prim[(prim['yahoo_iv'] > 0) & np.isfinite(prim['yahoo_iv'])].copy()
ivd['iv0'] = bsm_call(
    ivd['S'], ivd['K'], ivd['T_actact'], RISK_FREE, ivd['yahoo_iv'], 0.0
)
ivd['ivq'] = bsm_call(
    ivd['S'], ivd['K'], ivd['T_actact'], RISK_FREE, ivd['yahoo_iv'], ivd['q']
)

rows = []
for sym in TICKERS:
    g = ivd[ivd['symbol'] == sym]
    mae0 = (g['iv0'] - g['market_mid']).abs().mean()
    maeq = (g['ivq'] - g['market_mid']).abs().mean()
    corr = np.corrcoef(g['market_mid'], g['ivq'])[0, 1]
    rows.append([
        sym, len(g), mae0, maeq,
        100 * (mae0 - maeq) / mae0,
        corr
    ])

iv_diag = pd.DataFrame(rows, columns=[
    'Ticker', 'N', 'MAE q=0', 'MAE q>0',
    'Improvement %', 'Corr market vs q>0'
])
display(iv_diag)


## 9. American-style diagnostic: 100-step Cox–Ross–Rubinstein tree

The observed U.S. single-stock options are American-style, while the closed-form BSM formula is European. This diagnostic estimates the early-exercise component under the same HV20 and continuous-dividend inputs. It is used as a **diagnostic**, not as a replacement for the primary BSM comparison.


In [ ]:
def crr_batch(S, K, T, r, sigma, q, steps=100, american=True):
    S = np.asarray(S, float)
    K = np.asarray(K, float)
    T = np.asarray(T, float)
    sigma = np.asarray(sigma, float)
    q = np.asarray(q, float)

    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    disc = np.exp(-r * dt)
    p = (np.exp((r - q) * dt) - d) / (u - d)

    j = np.arange(steps + 1)
    terminal_S = (
        S[:, None]
        * (u[:, None] ** j[None, :])
        * (d[:, None] ** (steps - j)[None, :])
    )
    V = np.maximum(terminal_S - K[:, None], 0.0)

    for step in range(steps - 1, -1, -1):
        V = disc[:, None] * (
            p[:, None] * V[:, 1:step + 2]
            + (1 - p)[:, None] * V[:, :step + 1]
        )

        if american:
            jj = np.arange(step + 1)
            node_S = (
                S[:, None]
                * (u[:, None] ** jj[None, :])
                * (d[:, None] ** (step - jj)[None, :])
            )
            V = np.maximum(V, node_S - K[:, None])

    return V[:, 0]


american_rows = []
for sym in TICKERS:
    g = prim[prim['symbol'] == sym]

    euro_tree = crr_batch(
        g['S'], g['K'], g['T_actact'], RISK_FREE,
        g['sigma_hv20'], g['q'], steps=100, american=False
    )
    american_tree = crr_batch(
        g['S'], g['K'], g['T_actact'], RISK_FREE,
        g['sigma_hv20'], g['q'], steps=100, american=True
    )

    premium = np.maximum(american_tree - euro_tree, 0.0)
    american_mae = np.abs(american_tree - g['market_mid']).mean()
    mae_q0 = g['AE_q0'].mean()

    american_rows.append([
        sym,
        len(g),
        premium.mean(),
        np.quantile(premium, 0.95),
        premium.max(),
        american_mae,
        100 * (mae_q0 - american_mae) / mae_q0
    ])

american_diag = pd.DataFrame(american_rows, columns=[
    'Ticker', 'N', 'Mean early-exercise premium', 'P95 premium',
    'Max premium', 'American q>0 MAE', 'Improvement vs q=0 %'
])
display(american_diag)


## 10. Publication figures

### Figure 1 design

To make the primary figure easier to read:

- **market midpoints are points only**;
- **BSM `q=0` is a dashed line**;
- **BSM `q>0` is a solid line**;
- the legend explicitly shows the different marker/line styles;
- observations are plotted against normalized strike `K/S` and normalized option value `C/S`.

The figure is saved in both **PNG (300 dpi)** and **vector PDF** formats.


In [ ]:
# Figure 1 — improved, publication-ready normalized-price visualization
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.fontsize': 10
})

fig, axes = plt.subplots(3, 3, figsize=(12.6, 10.2))

for ax, sym in zip(axes.ravel(), TICKERS):
    g = prim[prim['symbol'] == sym].copy().sort_values('K_over_S')

    # Observed market values: points only, lightly transparent.
    ax.scatter(
        g['K_over_S'],
        g['market_mid'] / g['S'],
        s=14,
        alpha=0.35,
        marker='o',
        label='Market midpoint',
        rasterized=True
    )

    # No-dividend BSM: dashed line.
    ax.plot(
        g['K_over_S'],
        g['BS_q0'] / g['S'],
        linestyle='--',
        linewidth=2.0,
        label='BSM $q=0$'
    )

    # Dividend-adjusted BSM: solid line.
    ax.plot(
        g['K_over_S'],
        g['BSM_q'] / g['S'],
        linestyle='-',
        linewidth=2.2,
        label='BSM $q>0$'
    )

    ax.set_title(sym, pad=6)
    ax.set_xlabel('$K/S$')
    ax.set_ylabel('$C/S$')
    ax.set_xlim(0, 2.2)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.18)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 0.985),
    columnspacing=2.2,
    handlelength=2.8,
    handletextpad=0.7,
    borderpad=0.5
)
fig.suptitle(
    'Observed and model call values across all 13 trading days',
    y=0.94,
    fontsize=15
)
fig.tight_layout(rect=(0, 0, 1, 0.91))
fig.savefig(FIG / 'figure1_normalized_prices.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG / 'figure1_normalized_prices.pdf', bbox_inches='tight')
plt.show()

# Figure 2 — economic effect of dividend adjustment on MAE
fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.bar(pricing_summary['Ticker'], pricing_summary['MAE improvement %'])
ax.axhline(0, linewidth=1)
ax.set_ylabel('MAE improvement from dividend adjustment (%)')
ax.set_xlabel('Ticker')
ax.set_title('Effect of dividend adjustment on pricing accuracy (independent HV20)')
ax.grid(axis='y', alpha=0.18)
fig.tight_layout()
fig.savefig(FIG / 'figure2_mae_change.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG / 'figure2_mae_change.pdf', bbox_inches='tight')
plt.show()

# Figure 3 — analytical dividend-sensitivity approximation
fig, ax = plt.subplots(figsize=(6.2, 5.3))
ax.scatter(
    prim['dividend_omission_effect'],
    prim['dividend_approx'],
    s=10,
    alpha=0.30
)
lo = min(prim['dividend_omission_effect'].min(), prim['dividend_approx'].min())
hi = max(prim['dividend_omission_effect'].max(), prim['dividend_approx'].max())
ax.plot([lo, hi], [lo, hi], linewidth=1.2)
ax.set_xlabel('Exact BSM difference $C(0)-C(q)$')
ax.set_ylabel('First-order approximation')
ax.set_title('Dividend-sensitivity approximation to omission effect')
ax.grid(alpha=0.18)
fig.tight_layout()
fig.savefig(FIG / 'figure3_dividend_approximation.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG / 'figure3_dividend_approximation.pdf', bbox_inches='tight')
plt.show()


## 11. Export all reproducibility outputs

The notebook writes the cleaned primary dataset, analysis tables, figure files, and environment information under `revised_outputs/`.


In [ ]:
OUTPUT = ROOT / 'revised_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)

# Cleaned analysis dataset
prim.to_csv(OUTPUT / 'cleaned_primary_analysis.csv', index=False)

# Tables
sample_summary.to_csv(TAB / 'table1_sample_summary.csv', index=False)
pricing_summary.to_csv(TAB / 'table2_pricing_results.csv', index=False)
tests.to_csv(TAB / 'table3_paired_tests.csv', index=False)
robustness.to_csv(TAB / 'table4_robustness_long.csv', index=False)
robustness_pivot.to_csv(TAB / 'table4_robustness_pivot.csv')
iv_diag.to_csv(TAB / 'table5_yahoo_iv_diagnostic.csv', index=False)
american_diag.to_csv(TAB / 'table6_american_style_diagnostic.csv', index=False)
volatility_table.to_csv(TAB / 'historical_volatility_inputs.csv')
audit.to_csv(TAB / 'raw_data_audit.csv')

# Environment metadata
with open(OUTPUT / 'environment.txt', 'w', encoding='utf-8') as f:
    f.write(f'Python: {platform.python_version()}\n')
    f.write(f'NumPy: {np.__version__}\n')
    f.write(f'pandas: {pd.__version__}\n')
    f.write(f'SciPy: {scipy.__version__}\n')
    f.write(f'statsmodels: {statsmodels.__version__}\n')
    f.write(f'Risk-free rate: {RISK_FREE}\n')
    f.write(f'Expiry: {EXPIRY.date()}\n')
    f.write(f'Raw rows: {len(raw)}\n')
    f.write(f'Primary rows: {len(prim)}\n')
    f.write(f'Approximation correlation: {approx_corr:.9f}\n')
    f.write(f'Mean absolute approximation error: {mean_approx_abs_error:.9f}\n')

print('Saved outputs to:', OUTPUT)
for p in sorted(OUTPUT.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(ROOT))


## 12. Reproducibility smoke test

With the frozen replication data used for the revised manuscript, the expected core counts are:

- raw option-date observations: **9,959**;
- primary cleaned observations: **5,115**;
- exact/first-order dividend-omission-effect correlation: approximately **0.999996**.

A warning rather than a hard failure is used so that future versions of the repository can intentionally update the data without breaking the notebook.


In [ ]:
expected_raw = 9959
expected_primary = 5115
expected_corr = 0.999996339

checks = {
    'Raw row count': (len(raw), expected_raw),
    'Primary row count': (len(prim), expected_primary),
}

for label, (actual, expected) in checks.items():
    status = 'OK' if actual == expected else 'CHECK DATA VERSION'
    print(f'{label}: {actual} (expected {expected}) -> {status}')

print(
    f'Approximation correlation: {approx_corr:.9f} '
    f'(reference {expected_corr:.9f})'
)

if abs(approx_corr - expected_corr) > 1e-6:
    print('WARNING: approximation correlation differs from the frozen-data reference.')


## 13. Download regenerated results in Google Colab

Running the following cell in Colab creates one ZIP containing all exported figures, tables, the cleaned dataset, and environment information.


In [ ]:
archive_base = Path('/content/BSM_revision_results') if IN_COLAB else ROOT / 'BSM_revision_results'
archive_path = shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=ROOT / 'revised_outputs'
)
print('Created:', archive_path)

if IN_COLAB:
    from google.colab import files
    files.download(archive_path)
else:
    print('Local run: results ZIP saved at', archive_path)
